# BagZITboost — Optuna HPO + τ_π joint search (temp v2)

**목적**: BagZIT + π 임계값 (τ_π) 을 22번째 HP로 함께 탐색하여 ZIT 단독 대비 개선 검증.

**핵심 변경 (vs bag-zit-hpo-001)**:
- N_TRIALS: 10 → **30** (22차원 탐색에 맞춤)
- 22번째 HP **τ_π** 추가 ([0.5, 1.0]). τ_π=1.0 이면 사실상 임계값 off.
- 매 trial 의 objective 가 die-level π 에 τ_π 적용 → unit sum → RMSE 최소화.
- die-level CSV 저장 (π/μ/pred/pred_clipped/health) + unit raw + unit clipped 분리.
- Optuna HP importance 그룹별 분석 셀 추가.

**격리**: `4_output/_temp/bag_zit_hpo/` 출력. 기존 final/ 모듈 영향 없음.

## 1. 환경 + import

In [1]:
import os, sys

%run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess
from modules.zi_tweedie import ZITboostRegressor

import lightgbm as lgb
from sklearn.model_selection import KFold
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
optuna v4.7.0


## 2. 실험 설정

In [2]:
# ── 실험 식별 ──
EXP_ID  = 'bag-zit-hpo-002'   # 001 (τ_π 없는 버전) 과 분리
USER    = 'jh'
N_TRIALS = 30
N_FOLDS  = 5
CLIP_Y_EXTREME = True

# ── τ_π 학습 (22번째 HP) ──
USE_DIE_PI_THRESHOLD = True   # False면 τ_π=1.0 고정 (off)

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_hpo')
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

# ── 전처리 PARAMS (origin/main 01_zit_only 동일, '고정') ──
PARAMS = {
    'missing_threshold':          0.4,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.98,
    'post_impute_corr_keep_by':   'std',
}

print(f'EXP_ID={EXP_ID} | USER={USER} | N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS}')
print(f'USE_DIE_PI_THRESHOLD={USE_DIE_PI_THRESHOLD}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DB_PATH={DB_PATH}')

EXP_ID=bag-zit-hpo-002 | USER=jh | N_TRIALS=1 | N_FOLDS=5
USE_DIE_PI_THRESHOLD=True
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_hpo
DB_PATH=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_hpo\optuna_jh_bag-zit-hpo-002.db


## 3. 데이터 로드 + 전처리 (die-level)

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
y_train_die_broadcast = xs_train_die[KEY_COL].map(y_train_unit).values.astype(np.float64)

# numpy array 변환 (한 번만)
X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values

print(f'\n[전처리 완료]')
print(f'  train die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  feat_cols: {len(feat_cols_clean)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, d

## 4. BagZITboostRegressor 정의 (baseline 노트북과 동일)

In [4]:
class BagZITboostRegressor(ZITboostRegressor):
    """ZITboost + bag (unit) constraint via B3 allocation.

    fit(X, y, unit_id) — y는 broadcasted unit y per die.
    매 EM iter (≥1) 마다 same-unit dies 에 unit_y 비례 분배 (B3).
    """

    @staticmethod
    def _allocate_b3(unit_y_per_unit, contribution, inverse, n_units):
        contrib_sum_per_unit = np.zeros(n_units)
        np.add.at(contrib_sum_per_unit, inverse, contribution)
        contrib_sum_die = contrib_sum_per_unit[inverse]
        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        n_die_die = n_die_per_unit[inverse]
        share = np.where(
            contrib_sum_die > 1e-12,
            contribution / np.maximum(contrib_sum_die, 1e-12),
            1.0 / np.maximum(n_die_die, 1.0),
        )
        return unit_y_per_unit[inverse] * share

    def fit(self, X, y, unit_id):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        unit_id = np.asarray(unit_id)

        unique_units, first_idx, inverse = np.unique(
            unit_id, return_index=True, return_inverse=True
        )
        n_units = len(unique_units)
        unit_y_per_unit = y[first_idx]

        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        y_die_alloc = unit_y_per_unit[inverse] / np.maximum(n_die_per_unit[inverse], 1.0)

        pi_arr, mu_arr, phi_arr = self._initialize(X, y_die_alloc)
        self._phi_current = phi_arr

        self.em_history_ = []
        prev_rmse = np.inf
        em_iter = 0

        for em_iter in range(self.n_em_iters):
            if em_iter > 0:
                contribution = np.clip((1 - pi_arr) * mu_arr, 0, None)
                y_die_alloc = self._allocate_b3(
                    unit_y_per_unit, contribution, inverse, n_units
                )

            posterior = self._e_step(y_die_alloc, pi_arr, mu_arr, phi_arr)

            self._phi_current = phi_arr
            lgb_pi, lgb_mu, lgb_phi, pi_arr, mu_arr, phi_arr = \
                self._m_step(X, y_die_alloc, posterior)

            pred_die = np.clip((1 - pi_arr) * mu_arr, 0, None)
            pred_unit = np.zeros(n_units)
            np.add.at(pred_unit, inverse, pred_die)
            rmse_unit = float(np.sqrt(np.mean((unit_y_per_unit - pred_unit) ** 2)))

            self.em_history_.append({
                'iter':           em_iter + 1,
                'unit_rmse':      rmse_unit,
                'pi_mean':        float(pi_arr.mean()),
                'mu_mean':        float(mu_arr.mean()),
            })

            rmse_delta = prev_rmse - rmse_unit
            if em_iter >= 2 and abs(rmse_delta) < self.em_tol:
                break
            prev_rmse = rmse_unit

        self.n_em_iters_actual_ = em_iter + 1
        self.lgb_pi_ = lgb_pi
        self.lgb_mu_ = lgb_mu
        self.lgb_phi_ = lgb_phi
        self.fitted_ = True
        return self

    def predict_unit(self, X, unit_id):
        if not hasattr(self, 'fitted_'):
            raise ValueError('Model not fitted')
        unit_id = np.asarray(unit_id)
        unique_units, inverse = np.unique(unit_id, return_inverse=True)
        n_units = len(unique_units)
        pred_die = self.predict(X)
        pred_unit = np.zeros(n_units)
        np.add.at(pred_unit, inverse, pred_die)
        return pred_unit, unique_units


print('BagZITboostRegressor 정의 완료')

BagZITboostRegressor 정의 완료


## 5. Optuna 탐색공간 정의 (22 HP — ZIT top 20 ±10% + τ_π)

ZIT `optuna_jh_zit-final-100.db` top 20 trial 분포 + τ_π.

| HP | 적용 범위 | 비고 |
|---|---|---|
| zeta | [1.10, 1.30] | |
| n_em_iters | [10, 20] | BagZIT 하한 ↓ |
| mu_max_depth | [4, 6] | ZIT top 20 lock=5 |
| pi_max_depth | [7, 9] | ZIT top 20 lock=8 |
| phi_max_depth | [4, 6] | ZIT top 20 lock=5 |
| ... | 나머지 ZIT IQR ±10% | |
| **τ_π** | **[0.5, 1.0]** | **π > τ_π 인 die의 pred=0. τ_π=1.0 이면 off.** |

In [5]:
def suggest_bag_zit_params(trial):
    """ZIT top 20 anchor 기반 narrow 탐색공간 + τ_π.

    Returns
    -------
    (model_params, tau_pi) : (dict, float)
        model_params 는 BagZIT 에 전달, tau_pi 는 die 후처리용.
    """
    params = dict(
        # ZIT 전용
        zeta=trial.suggest_float('zeta', 1.05, 1.50),
        n_em_iters=trial.suggest_int('n_em_iters', 8, 25),
        em_tol=1e-7,
        # μ (9개)
        mu_n_estimators=trial.suggest_int('mu_n_estimators', 100, 500),
        mu_learning_rate=trial.suggest_float('mu_learning_rate', 0.003, 0.03, log=True),
        mu_num_leaves=trial.suggest_int('mu_num_leaves', 60, 256),
        mu_max_depth=trial.suggest_int('mu_max_depth', 3, 8),
        mu_min_child_samples=trial.suggest_int('mu_min_child_samples', 20, 150),
        mu_subsample=trial.suggest_float('mu_subsample', 0.50, 0.95),
        mu_colsample_bytree=trial.suggest_float('mu_colsample_bytree', 0.20, 0.70),
        mu_reg_alpha=trial.suggest_float('mu_reg_alpha', 1e-6, 1e-1, log=True),
        mu_reg_lambda=trial.suggest_float('mu_reg_lambda', 1e-3, 1.0, log=True),
        # π (5개)
        pi_n_estimators=trial.suggest_int('pi_n_estimators', 100, 500),
        pi_learning_rate=trial.suggest_float('pi_learning_rate', 0.02, 0.20, log=True),
        pi_num_leaves=trial.suggest_int('pi_num_leaves', 30, 200),
        pi_max_depth=trial.suggest_int('pi_max_depth', 5, 12),
        pi_min_child_samples=trial.suggest_int('pi_min_child_samples', 10, 100),
        # φ (5개)
        phi_n_estimators=trial.suggest_int('phi_n_estimators', 30, 200),
        phi_learning_rate=trial.suggest_float('phi_learning_rate', 0.005, 0.05, log=True),
        phi_num_leaves=trial.suggest_int('phi_num_leaves', 30, 150),
        phi_max_depth=trial.suggest_int('phi_max_depth', 3, 8),
        phi_min_child_samples=trial.suggest_int('phi_min_child_samples', 10, 200),
        # 공통
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
        device='cpu',
    )

    # τ_π (22번째 HP)
    if USE_DIE_PI_THRESHOLD:
        tau_pi = trial.suggest_float('tau_pi', 0.5, 1.0)
    else:
        tau_pi = 1.0   # off (no die killed)

    return params, tau_pi


print(f'탐색공간 {"22" if USE_DIE_PI_THRESHOLD else "21"} HP 정의 완료'
      + (' (τ_π 포함, [0.5, 1.0])' if USE_DIE_PI_THRESHOLD else ''))

탐색공간 22 HP 정의 완료 (τ_π 포함, [0.5, 1.0])


## 6. K-fold objective

각 trial:
1. 22 HP suggest (τ_π 포함)
2. 5-fold OOF (unit-level KFold, leakage 방지)
3. fold마다: 학습 → die-level π/μ → τ_π 적용 → unit sum → RMSE
4. OOF unit RMSE (clipped 기준) 반환

In [6]:
import time

unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))


def _sum_die_to_unit(pred_die, uid_die):
    """die-level pred 를 unit별 합산. (pred_unit, unique_units) 반환."""
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    n_units = len(unique_units)
    pred_unit = np.zeros(n_units)
    np.add.at(pred_unit, inverse, pred_die)
    return pred_unit, unique_units


def _apply_tau_pi(pred_die, pi_die, tau_pi):
    """π > τ_π 인 die 의 pred 를 0 으로."""
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def objective(trial):
    params, tau_pi = suggest_bag_zit_params(trial)

    oof_unit_pred  = pd.Series(np.nan, index=y_train_unit.index)
    fold_preds_val = []
    fold_preds_test = []
    fold_oof_rmse  = []

    t0 = time.time()

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)

        X_tr   = X_train_die[tr_die_mask]
        X_vl   = X_train_die[vl_die_mask]
        y_tr   = y_train_die_broadcast[tr_die_mask]
        uid_tr = uid_train_die[tr_die_mask]
        uid_vl = uid_train_die[vl_die_mask]

        model = BagZITboostRegressor(**params)
        model.fit(X_tr, y_tr, unit_id=uid_tr)

        # OOF — die-level π/μ → τ_π 적용 → unit sum
        pi_vl, mu_vl, _ = model.predict_components(X_vl)
        pred_die_vl_raw = np.clip((1 - pi_vl) * mu_vl, 0, None)
        pred_die_vl = _apply_tau_pi(pred_die_vl_raw, pi_vl, tau_pi)
        pred_unit_vl, ids_vl = _sum_die_to_unit(pred_die_vl, uid_vl)

        oof_unit_pred.loc[ids_vl] = pred_unit_vl
        y_vl_true = y_train_unit.loc[ids_vl].values
        fold_rmse = float(np.sqrt(np.mean((pred_unit_vl - y_vl_true) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        # holdout val/test — 동일 방식
        pi_v, mu_v, _ = model.predict_components(X_val_die)
        pred_die_v_raw = np.clip((1 - pi_v) * mu_v, 0, None)
        pred_die_v = _apply_tau_pi(pred_die_v_raw, pi_v, tau_pi)
        pred_unit_val, ids_val = _sum_die_to_unit(pred_die_v, uid_val_die)

        pi_t, mu_t, _ = model.predict_components(X_test_die)
        pred_die_t_raw = np.clip((1 - pi_t) * mu_t, 0, None)
        pred_die_t = _apply_tau_pi(pred_die_t_raw, pi_t, tau_pi)
        pred_unit_test, ids_test = _sum_die_to_unit(pred_die_t, uid_test_die)

        fold_preds_val.append(pd.Series(pred_unit_val, index=ids_val))
        fold_preds_test.append(pd.Series(pred_unit_test, index=ids_test))

        # Pruning report (fold 평균 RMSE)
        avg_so_far = float(np.mean(fold_oof_rmse))
        trial.report(avg_so_far, step=fold_idx)
        if trial.should_prune():
            elapsed = time.time() - t0
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', elapsed)
            trial.set_user_attr('tau_pi', tau_pi)
            raise optuna.TrialPruned()

    # 전체 RMSE
    val_unit_pred  = pd.concat(fold_preds_val,  axis=1).mean(axis=1).reindex(y_val_unit.index)
    test_unit_pred = pd.concat(fold_preds_test, axis=1).mean(axis=1).reindex(y_test_unit.index)

    if oof_unit_pred.isna().any():
        raise RuntimeError('OOF NaN')

    oof_rmse  = float(np.sqrt(np.mean((oof_unit_pred.values  - y_train_unit.values) ** 2)))
    val_rmse  = float(np.sqrt(np.mean((val_unit_pred.values  - y_val_unit.values)  ** 2)))
    test_rmse = float(np.sqrt(np.mean((test_unit_pred.values - y_test_unit.values) ** 2)))

    elapsed = time.time() - t0
    trial.set_user_attr('val_rmse',     val_rmse)
    trial.set_user_attr('test_rmse',    test_rmse)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    trial.set_user_attr('elapsed_sec',  elapsed)
    trial.set_user_attr('tau_pi',       tau_pi)

    print(f'  trial #{trial.number}: τ_π={tau_pi:.3f}, oof={oof_rmse:.6f}, '
          f'val={val_rmse:.6f}, test={test_rmse:.6f}, t={elapsed:.0f}s')

    return oof_rmse


print(f'fold split 고정: {N_FOLDS} folds, {len(FOLDS)} 개')

fold split 고정: 5 folds, 5 개


## 7. Optuna study 실행

샘플러: **TPE multivariate** (HP joint 분포 학습 — 22 HP 상호작용)

Pruner: **MedianPruner** (n_startup_trials=4, n_warmup_steps=2 — fold 2 이후 prune 가능)

DB: SQLite (재개 가능). 같은 study 이름으로 재실행 시 trial 이어서 진행.

In [7]:
sampler = TPESampler(
    multivariate=True,    # joint 분포 (22 HP 상호작용 학습)
    group=True,           # (조건부 search space 없으므로 사실상 무효, 무해)
    n_startup_trials=4,   # 4 random + 26 multivariate exploit
    seed=SEED,
)
pruner = MedianPruner(
    n_startup_trials=4,
    n_warmup_steps=2,
)

study_meta = {
    'exp_id':              EXP_ID,
    'user':                USER,
    'model':               'BagZITboost (안 C / B3) + τ_π joint search',
    'n_trials':            N_TRIALS,
    'n_folds':             N_FOLDS,
    'use_die_pi_threshold': USE_DIE_PI_THRESHOLD,
    'tau_pi_range':        '[0.5, 1.0]' if USE_DIE_PI_THRESHOLD else 'off',
    'sampler':             'TPE multivariate group=True n_startup=4',
    'pruner':              'MedianPruner n_startup=4 n_warmup=2',
    'search_space_anchor': 'ZIT top 20 (optuna_jh_zit-final-100.db) ±10% + tau_pi',
    'preprocess_PARAMS':   PARAMS,
    'effective_pp_params': pp['effective_params'],
    'CLIP_Y_EXTREME':      CLIP_Y_EXTREME,
    'feat_cols_clean_n':   len(feat_cols_clean),
    'SEED':                int(SEED),
}

study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler,
    pruner=pruner,
    direction='minimize',
    load_if_exists=True,   # 재실행 시 이어서
)
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}, DB: {DB_PATH}')
print(f'기존 trial 수: {len(study.trials)}')

t_total = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_total:.0f}s, total trials={len(study.trials)}')
print(f'  best OOF RMSE: {study.best_value:.6f}')

[I 2026-04-30 13:50:25,907] A new study created in RDB with name: bag-zit-hpo-002


study: bag-zit-hpo-002, DB: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_hpo\optuna_jh_bag-zit-hpo-002.db
기존 trial 수: 0


  0%|          | 0/1 [00:00<?, ?it/s]

  trial #0: τ_π=0.570, oof=0.005518, val=0.005717, test=0.008419, t=1212s
[I 2026-04-30 14:10:38,360] Trial 0 finished with value: 0.005518126054452075 and parameters: {'zeta': 1.1749080237694725, 'n_em_iters': 20, 'mu_n_estimators': 247, 'mu_learning_rate': 0.008444747915787608, 'mu_num_leaves': 121, 'mu_max_depth': 4, 'mu_min_child_samples': 62, 'mu_subsample': 0.7232352291549871, 'mu_colsample_bytree': 0.42022300234864174, 'mu_reg_alpha': 0.0013035123791853842, 'mu_reg_lambda': 0.010636066512540286, 'pi_n_estimators': 227, 'pi_learning_rate': 0.09179681421265247, 'pi_num_leaves': 64, 'pi_max_depth': 7, 'pi_min_child_samples': 26, 'phi_n_estimators': 58, 'phi_learning_rate': 0.014386906671916194, 'phi_num_leaves': 72, 'phi_max_depth': 4, 'phi_min_child_samples': 69, 'tau_pi': 0.569746930326021}. Best is trial 0 with value: 0.005518126054452075.

[HPO 완료] 전체 1212s, total trials=1
  best OOF RMSE: 0.005518


## 8. Best params + trial history

In [8]:
import matplotlib.pyplot as plt

best_trial = study.best_trial
best_params = best_trial.params

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE  : {best_trial.value:.6f}')
print(f'  val RMSE  : {best_trial.user_attrs.get("val_rmse", "N/A")}')
print(f'  test RMSE : {best_trial.user_attrs.get("test_rmse", "N/A")}')
print(f'  best τ_π  : {best_trial.user_attrs.get("tau_pi", "N/A")}')
print(f'  elapsed   : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')
print(f'\n  best params:')
for k, v in sorted(best_params.items()):
    print(f'    {k}: {v}')

# trial history
df_trials = pd.DataFrame([
    {
        'trial':     t.number,
        'state':     t.state.name,
        'oof_rmse':  t.value,
        'val_rmse':  t.user_attrs.get('val_rmse'),
        'test_rmse': t.user_attrs.get('test_rmse'),
        'tau_pi':    t.user_attrs.get('tau_pi'),
        'elapsed':   t.user_attrs.get('elapsed_sec'),
    }
    for t in study.trials
])
print('\n=== trial history ===')
print(df_trials.to_string(index=False))

# plot
complete = df_trials[df_trials['state'] == 'COMPLETE']
if len(complete) > 1:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(complete['trial'], complete['oof_rmse'], marker='o', label='OOF')
    if complete['val_rmse'].notna().any():
        ax.plot(complete['trial'], complete['val_rmse'], marker='s', alpha=0.7, label='val')
    ax.axhline(study.best_value, color='red', linestyle='--', alpha=0.5, label=f'best={study.best_value:.6f}')
    ax.set_xlabel('trial')
    ax.set_ylabel('unit RMSE')
    ax.set_title('Optuna trial history')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

=== Best Trial #0 ===
  OOF RMSE  : 0.005518
  val RMSE  : 0.005717333420591509
  test RMSE : 0.008418828225694647
  best τ_π  : 0.569746930326021
  elapsed   : 1212s

  best params:
    mu_colsample_bytree: 0.42022300234864174
    mu_learning_rate: 0.008444747915787608
    mu_max_depth: 4
    mu_min_child_samples: 62
    mu_n_estimators: 247
    mu_num_leaves: 121
    mu_reg_alpha: 0.0013035123791853842
    mu_reg_lambda: 0.010636066512540286
    mu_subsample: 0.7232352291549871
    n_em_iters: 20
    phi_learning_rate: 0.014386906671916194
    phi_max_depth: 4
    phi_min_child_samples: 69
    phi_n_estimators: 58
    phi_num_leaves: 72
    pi_learning_rate: 0.09179681421265247
    pi_max_depth: 7
    pi_min_child_samples: 26
    pi_n_estimators: 227
    pi_num_leaves: 64
    tau_pi: 0.569746930326021
    zeta: 1.1749080237694725

=== trial history ===
 trial    state  oof_rmse  val_rmse  test_rmse   tau_pi     elapsed
     0 COMPLETE  0.005518  0.005717   0.008419 0.569747 1212.1353

## 8.5 HP importance + 그룹별 분석

fANOVA 기반 단일 HP importance + μ/π/φ/ZIT/τ_π 그룹별 합산.
**N_TRIALS 가 적으면 importance 계산 실패 가능** (≥10 권장).

In [9]:
import optuna.importance as opt_imp

try:
    imp = opt_imp.get_param_importances(study)

    print('=== HP importance (fANOVA, 상위 12개) ===')
    for i, (k, v) in enumerate(sorted(imp.items(), key=lambda x: -x[1])[:12]):
        bar = '█' * int(v * 50)
        print(f'  {i+1:2d}. {k:30s}: {v:.4f}  {bar}')

    # μ/π/φ/ZIT/τ_π 그룹별 합산
    groups = {'μ (회귀)': 0.0, 'π (분류)': 0.0, 'φ (분산)': 0.0, 'ZIT(zeta/n_em)': 0.0, 'τ_π': 0.0}
    for k, v in imp.items():
        if k.startswith('mu_'):
            groups['μ (회귀)'] += v
        elif k.startswith('pi_'):
            groups['π (분류)'] += v
        elif k.startswith('phi_'):
            groups['φ (분산)'] += v
        elif k == 'tau_pi':
            groups['τ_π'] += v
        else:
            groups['ZIT(zeta/n_em)'] += v

    print('\n=== 그룹별 importance 합산 (RMSE 에 대한 기여도) ===')
    for g, v in sorted(groups.items(), key=lambda x: -x[1]):
        bar = '█' * int(v * 50)
        print(f'  {g:18s}: {v:.4f}  {bar}')

    # plotly 시각화 (가능하면)
    try:
        import optuna.visualization as ov
        ov.plot_param_importances(study).show()
        ov.plot_parallel_coordinate(study).show()
        top2 = list(sorted(imp.items(), key=lambda x: -x[1]))[:2]
        if len(top2) >= 2:
            ov.plot_contour(study, params=[top2[0][0], top2[1][0]]).show()
    except Exception as e:
        print(f'\nplotly 시각화 스킵: {e}')

except (RuntimeError, ValueError) as e:
    print(f'importance 분석 스킵: {e}')
    print('  → trial 수가 적거나 모든 trial 이 같은 값일 가능성. N_TRIALS 늘리면 해결됨.')

importance 분석 스킵: Cannot evaluate parameter importances with only a single trial.
  → trial 수가 적거나 모든 trial 이 같은 값일 가능성. N_TRIALS 늘리면 해결됨.


## 9. Best params 5-fold refit + die-level 캡처

best params 로 다시 5-fold 학습하면서 **die-level π/μ/pred** 모두 캡처.

In [10]:
best_oof_rmse  = float(best_trial.value)
best_val_rmse  = float(best_trial.user_attrs.get('val_rmse', np.nan))
best_test_rmse = float(best_trial.user_attrs.get('test_rmse', np.nan))
best_tau_pi    = float(best_trial.user_attrs.get('tau_pi', 1.0))

print('=== Best (5-fold OOF + holdout, τ_π 적용 RMSE) ===')
print(f'  OOF unit RMSE  : {best_oof_rmse:.6f}')
print(f'  val unit RMSE  : {best_val_rmse:.6f}')
print(f'  test unit RMSE : {best_test_rmse:.6f}')
print(f'  best τ_π       : {best_tau_pi:.4f}')

# best params 정리 (tau_pi 는 model 에 안 들어감)
best_params_model = {k: v for k, v in best_params.items() if k != 'tau_pi'}
best_full_params = dict(
    **best_params_model,
    em_tol=1e-7,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
    device='cpu',
)

# die-level 캡처용 array
n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)

oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)

val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)

test_die_pi   = np.zeros(n_test_die)
test_die_mu   = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

print('\n=== best params 로 5-fold refit (die-level 캡처) ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_die_mask = np.isin(uid_train_die, tr_units)
    vl_die_mask = np.isin(uid_train_die, vl_units)
    X_tr, y_tr = X_train_die[tr_die_mask], y_train_die_broadcast[tr_die_mask]
    X_vl       = X_train_die[vl_die_mask]
    uid_tr     = uid_train_die[tr_die_mask]

    model = BagZITboostRegressor(**best_full_params)
    model.fit(X_tr, y_tr, unit_id=uid_tr)

    # OOF die-level π/μ/pred
    pi_vl, mu_vl, _ = model.predict_components(X_vl)
    pred_vl = np.clip((1 - pi_vl) * mu_vl, 0, None)
    oof_die_pi[vl_die_mask]   = pi_vl
    oof_die_mu[vl_die_mask]   = mu_vl
    oof_die_pred[vl_die_mask] = pred_vl

    # val/test (5-fold avg)
    pi_v, mu_v, _ = model.predict_components(X_val_die)
    pi_t, mu_t, _ = model.predict_components(X_test_die)
    val_die_pi   += pi_v / N_FOLDS
    val_die_mu   += mu_v / N_FOLDS
    val_die_pred += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi   += pi_t / N_FOLDS
    test_die_mu   += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

# NaN 검증
assert not np.isnan(oof_die_pi).any(),   'OOF die π 미커버'
assert not np.isnan(oof_die_mu).any(),   'OOF die μ 미커버'
assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

print('\n[refit 완료] die-level π/μ/pred 캡처 OK')
print(f'  oof_die: {n_train_die}, val_die: {n_val_die}, test_die: {n_test_die}')

=== Best (5-fold OOF + holdout, τ_π 적용 RMSE) ===
  OOF unit RMSE  : 0.005518
  val unit RMSE  : 0.005717
  test unit RMSE : 0.008419
  best τ_π       : 0.5697

=== best params 로 5-fold refit (die-level 캡처) ===
  fold 1/5 done (265s)
  fold 2/5 done (506s)
  fold 3/5 done (746s)
  fold 4/5 done (984s)
  fold 5/5 done (1224s)

[refit 완료] die-level π/μ/pred 캡처 OK
  oof_die: 104748, val_die: 34908, test_die: 34916


## 10. τ_π 적용 + raw vs clipped 비교

best τ_π 를 die-level π 에 적용하여 clipped 예측 만들고 unit 합산.
raw (τ_π 미적용) 과 clipped (τ_π 적용) 양쪽 RMSE 비교.

In [11]:
# τ_π 적용 (die-level)
oof_die_pred_clipped  = np.where(oof_die_pi  > best_tau_pi, 0.0, oof_die_pred)
val_die_pred_clipped  = np.where(val_die_pi  > best_tau_pi, 0.0, val_die_pred)
test_die_pred_clipped = np.where(test_die_pi > best_tau_pi, 0.0, test_die_pred)

# unit aggregate
oof_unit_raw_arr,  oof_unit_ids  = _sum_die_to_unit(oof_die_pred,         uid_train_die)
oof_unit_clip_arr, _              = _sum_die_to_unit(oof_die_pred_clipped, uid_train_die)
val_unit_raw_arr,  val_unit_ids  = _sum_die_to_unit(val_die_pred,         uid_val_die)
val_unit_clip_arr, _              = _sum_die_to_unit(val_die_pred_clipped, uid_val_die)
test_unit_raw_arr, test_unit_ids = _sum_die_to_unit(test_die_pred,        uid_test_die)
test_unit_clip_arr, _             = _sum_die_to_unit(test_die_pred_clipped, uid_test_die)

# Series 로 (재정렬)
oof_unit_raw   = pd.Series(oof_unit_raw_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
oof_unit_clip  = pd.Series(oof_unit_clip_arr, index=oof_unit_ids).reindex(y_train_unit.index)
val_unit_raw   = pd.Series(val_unit_raw_arr,  index=val_unit_ids).reindex(y_val_unit.index)
val_unit_clip  = pd.Series(val_unit_clip_arr, index=val_unit_ids).reindex(y_val_unit.index)
test_unit_raw  = pd.Series(test_unit_raw_arr, index=test_unit_ids).reindex(y_test_unit.index)
test_unit_clip = pd.Series(test_unit_clip_arr, index=test_unit_ids).reindex(y_test_unit.index)

# RMSE
def _rmse(pred, true):
    return float(np.sqrt(np.mean((pred.values - true.values) ** 2)))

oof_rmse_raw      = _rmse(oof_unit_raw,   y_train_unit)
oof_rmse_clipped  = _rmse(oof_unit_clip,  y_train_unit)
val_rmse_raw      = _rmse(val_unit_raw,   y_val_unit)
val_rmse_clipped  = _rmse(val_unit_clip,  y_val_unit)
test_rmse_raw     = _rmse(test_unit_raw,  y_test_unit)
test_rmse_clipped = _rmse(test_unit_clip, y_test_unit)

# 비교 출력
print('=' * 75)
print(f'  best τ_π = {best_tau_pi:.4f}'
      + ('  (≈ off)' if best_tau_pi >= 0.99 else ''))
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"Raw":12s}  {oof_rmse_raw:11.6f}  {val_rmse_raw:11.6f}  {test_rmse_raw:11.6f}')
print(f'  {"Clipped":12s}  {oof_rmse_clipped:11.6f}  {val_rmse_clipped:11.6f}  {test_rmse_clipped:11.6f}')
print(f'  {"Δ":12s}  {oof_rmse_clipped-oof_rmse_raw:+11.6f}  '
      f'{val_rmse_clipped-val_rmse_raw:+11.6f}  '
      f'{test_rmse_clipped-test_rmse_raw:+11.6f}')
print('=' * 75)

if val_rmse_clipped > val_rmse_raw and test_rmse_clipped > test_rmse_raw:
    print('  ⚠ val/test 모두 악화 — τ_π 적용 비추천')
elif val_rmse_clipped < val_rmse_raw and test_rmse_clipped < test_rmse_raw:
    print('  ✅ val/test 모두 개선 — τ_π 채택 권장')
else:
    print('  ⚠ val/test 결과 엇갈림 — 신중 판단 필요')

# 0 처리된 die 비율
killed_oof  = float((oof_die_pi  > best_tau_pi).mean())
killed_val  = float((val_die_pi  > best_tau_pi).mean())
killed_test = float((test_die_pi > best_tau_pi).mean())
print(f'\n  τ_π 로 0 처리된 die 비율: '
      f'OOF={killed_oof:.1%}, val={killed_val:.1%}, test={killed_test:.1%}')

  best τ_π = 0.5697
                        OOF          val         test
  Raw              0.005504     0.005709     0.008412
  Clipped          0.005518     0.005721     0.008421
  Δ               +0.000014    +0.000012    +0.000009
  ⚠ val/test 모두 악화 — τ_π 적용 비추천

  τ_π 로 0 처리된 die 비율: OOF=25.6%, val=23.8%, test=25.1%


## 11. 아티팩트 저장

In [12]:
import json

# ── die-level CSV ──
def _build_die_df(uid_arr, die_id_arr, position_arr, pi, mu, pred, pred_clipped, y_unit):
    df = pd.DataFrame({
        KEY_COL:        uid_arr,
        DIE_KEY_COL:    die_id_arr,
        'position':     position_arr,
        'pi':           pi,
        'one_minus_pi': 1.0 - pi,
        'mu':           mu,
        'pred':         pred,
        'pred_clipped': pred_clipped,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

oof_die_df = _build_die_df(
    uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
    oof_die_pi, oof_die_mu, oof_die_pred, oof_die_pred_clipped, y_train_unit,
)
val_die_df = _build_die_df(
    uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
    val_die_pi, val_die_mu, val_die_pred, val_die_pred_clipped, y_val_unit,
)
test_die_df = _build_die_df(
    uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
    test_die_pi, test_die_mu, test_die_pred, test_die_pred_clipped, y_test_unit,
)
oof_die_df.to_csv(os.path.join(OUT_DIR,  'oof_die.csv'),  index=False)
val_die_df.to_csv(os.path.join(OUT_DIR,  'val_die.csv'),  index=False)
test_die_df.to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

# ── unit-level CSV (raw) ──
def _build_unit_df(unit_pred, y_unit):
    return pd.DataFrame({
        KEY_COL:  unit_pred.index.values,
        'pred':   unit_pred.values,
        'health': y_unit.reindex(unit_pred.index).values,
    })

_build_unit_df(oof_unit_raw,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit_raw,  y_val_unit  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit_raw, y_test_unit ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# ── unit-level CSV (clipped, best τ_π 적용) ──
_build_unit_df(oof_unit_clip,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit_clipped.csv'),  index=False)
_build_unit_df(val_unit_clip,  y_val_unit  ).to_csv(os.path.join(OUT_DIR, 'val_unit_clipped.csv'),  index=False)
_build_unit_df(test_unit_clip, y_test_unit ).to_csv(os.path.join(OUT_DIR, 'test_unit_clipped.csv'), index=False)

# ── best params + meta ──
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'best_trial_number': best_trial.number,
        'best_oof_rmse':     best_oof_rmse,
        'best_val_rmse':     best_val_rmse,
        'best_test_rmse':    best_test_rmse,
        'best_tau_pi':       best_tau_pi,
        'best_params':       best_params,
    }, f, indent=2, ensure_ascii=False, default=str)

meta = {
    'exp_id':              EXP_ID,
    'model':               'BagZITboost (안 C / B3) + τ_π joint search',
    'use_die_pi_threshold': USE_DIE_PI_THRESHOLD,
    'best_tau_pi':         best_tau_pi,
    'n_trials_run':        len(study.trials),
    'n_trials_complete':   sum(1 for t in study.trials if t.state.name == 'COMPLETE'),
    'n_folds':             N_FOLDS,
    # raw RMSE (τ_π 미적용)
    'raw_oof_rmse':        oof_rmse_raw,
    'raw_val_rmse':        val_rmse_raw,
    'raw_test_rmse':       test_rmse_raw,
    # clipped RMSE (best τ_π 적용)
    'clipped_oof_rmse':    oof_rmse_clipped,
    'clipped_val_rmse':    val_rmse_clipped,
    'clipped_test_rmse':   test_rmse_clipped,
    # study best (objective 가 clipped RMSE)
    'study_best_oof_rmse': best_oof_rmse,
    'study_best_val_rmse': best_val_rmse,
    'study_best_test_rmse': best_test_rmse,
    'preprocess_PARAMS':   PARAMS,
    'effective_pp_params': pp['effective_params'],
    'CLIP_Y_EXTREME':      CLIP_Y_EXTREME,
    'feat_cols_clean_n':   len(feat_cols_clean),
    'SEED':                int(SEED),
    'sampler':             'TPE multivariate group=True n_startup=4',
    'tau_pi_range':        '[0.5, 1.0]' if USE_DIE_PI_THRESHOLD else 'off',
    'search_space_anchor': 'ZIT top 20 (optuna_jh_zit-final-100.db) ±10%',
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:35s}  {sz:10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_hpo
  best_params.json                            0.9 KB
  meta.json                                   1.7 KB
  oof_die.csv                            13,761.5 KB
  oof_unit.csv                              971.2 KB
  oof_unit_clipped.csv                      880.8 KB
  optuna_jh_bag-zit-hpo-001.db              112.0 KB
  optuna_jh_bag-zit-hpo-002.db              112.0 KB
  test_die.csv                            4,589.7 KB
  test_unit.csv                             323.7 KB
  test_unit_clipped.csv                     292.0 KB
  val_die.csv                             4,596.7 KB
  val_unit.csv                              323.7 KB
  val_unit_clipped.csv                      293.8 KB


## 12. 요약

In [13]:
print('=' * 75)
print(' BagZITboost HPO + τ_π Joint Search — 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  N_TRIALS run      : {len(study.trials)}')
print(f'  완료 trial        : {sum(1 for t in study.trials if t.state.name == "COMPLETE")}')
print(f'  Best trial        : #{best_trial.number}')
print(f'  Best τ_π          : {best_tau_pi:.4f}'
      + ('  (≈ off)' if best_tau_pi >= 0.99 else ''))
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"Raw":10s}  {oof_rmse_raw:11.6f}  {val_rmse_raw:11.6f}  {test_rmse_raw:11.6f}')
print(f'  {"Clipped":10s}  {oof_rmse_clipped:11.6f}  {val_rmse_clipped:11.6f}  {test_rmse_clipped:11.6f}')
print(f'  {"Δ":10s}  {oof_rmse_clipped-oof_rmse_raw:+11.6f}  '
      f'{val_rmse_clipped-val_rmse_raw:+11.6f}  '
      f'{test_rmse_clipped-test_rmse_raw:+11.6f}')
print('-' * 75)
print(f'  비교 기준선:')
print(f'    BagZIT baseline (HP fixed):    OOF=0.005524, val=0.005729, test=0.008428')
print(f'    ZIT 단독 zit-final-100:        OOF=0.005501 (180 trials)')
print(f'    ZIT 단독 zit-final-999:        OOF=0.005592, val=0.005772, test=0.008450 (1 trial)')
print(f'    BagZIT HPO 001 (τ_π 없음):     이전 실험 참고')
print('=' * 75)
print(f'  → BagZIT (clipped) 가 ZIT best 보다 낮으면 bag + τ_π 가설 검증 ✓')

 BagZITboost HPO + τ_π Joint Search — 결과 요약
  EXP_ID            : bag-zit-hpo-002
  N_TRIALS run      : 1
  완료 trial        : 1
  Best trial        : #0
  Best τ_π          : 0.5697
---------------------------------------------------------------------------
                      OOF          val         test
  Raw            0.005504     0.005709     0.008412
  Clipped        0.005518     0.005721     0.008421
  Δ             +0.000014    +0.000012    +0.000009
---------------------------------------------------------------------------
  비교 기준선:
    BagZIT baseline (HP fixed):    OOF=0.005524, val=0.005729, test=0.008428
    ZIT 단독 zit-final-100:        OOF=0.005501 (180 trials)
    ZIT 단독 zit-final-999:        OOF=0.005592, val=0.005772, test=0.008450 (1 trial)
    BagZIT HPO 001 (τ_π 없음):     이전 실험 참고
  → BagZIT (clipped) 가 ZIT best 보다 낮으면 bag + τ_π 가설 검증 ✓
